# 6. Comparación Final y Conclusiones

### Problema A - Regresión (PAY_AMT4)

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Regresión Lineal | 0.1544 | 11,765 | 4,811 |
| XGBoost | 0.5979 | 8,113 | 2,205 |


El modelo XGBoost superó ampliamente a la Regresión Lineal en todas las métricas. Un R² de 0.60 es un resultado sólido considerando que `PAY_AMT4` tiene una distribución muy sesgada con una gran concentración de valores cercanos a cero, lo que representa un reto inherente para cualquier modelo de regresión.

La variable más importante resultó ser `PAY_5` (estatus de pago en mayo), lo que tienesentido de negocio: el comportamiento de pago más reciente es el mejor predictor del monto que pagará el cliente el siguiente mes.


### Problema B - Clasificación (default.payment.next.month)

| Modelo | Accuracy | ROC-AUC | F1-Score | GINI |
|---|---|---|---|---|
| Regresión Logística | 0.7480 | 0.7118 | 0.4930 | 0.4236 |
| XGBoost | 0.7625 | 0.7774 | 0.5351 | 0.5547 |

XGBoost fue el mejor modelo también en clasificación. Un ROC-AUC de 0.777 y un GINI de 0.555 son resultados considerados buenos en la industria financiera para modelos de riesgo crediticio.

El dataset presentaba un desbalance de clases importante (77.9% no default vs 22.1% default), lo cual fue considerado en el modelado mediante el parámetro `scale_pos_weight` en XGBoost  y `class_weight="balanced"` en la Regresión Logística.

Las variables más importantes fueron `MAX_RETRASO` y `MESES_RETRASO`, ambas creadas en el feature engineering, lo que valida que el historial de retrasos es el predictor más poderoso de default crediticio.



### Conclusiones de Negocio

- El comportamiento de pago pasado es el mejor predictor tanto del monto de pag futuro como del riesgo de default.Un cliente que ha retrasado pagos múltiples meses tiene una probabilidad significativamente mayor de incumplir.

- El límite de crédito (LIMIT_BAL) tiene correlación negativa con el default, lo que sugiere que los clientes con mayor límite tienden a ser más responsables financieramente o tienen mayor capacidad de pago.

- Las variables demográficas (sexo, educación, estado civil) tienen poco poder predictiv, comparadas con el historial financiero, lo que indica que el comportamiento es más relevante que el perfil demográfico para evaluar riesgo crediticio.

- El modelo de clasificación puede ser utilizado por el banco para identificar clientes de alto riesgo y tomar acciones preventivas como ajuste de límites.


### Respuestas a las preguntas de la prueba

1. De las deficiencias en los datos, ¿cuáles y cómo las identificaste?

Se identificaron las siguientes deficiencias durante el proceso de limpieza:
- **EDUCATION**: 14 registros con valor 0, no documentado en el diccionario. Identificado mediante `value_counts()` y comparación contra los valores esperados {1,2,3,4,5,6}.
- **MARRIAGE**: 54 registros con valor 0, no documentado en el diccionario. Identificado con el mismo proceso. Ambos valores fueron marcados como NaN.
- **Columnas PAY_0 a PAY_6**: contienen valores -2 que no están en el diccionario original (el cual indica rango de -1 a 9). Se identificó mediante `unique()` en cada columna.
  Asumimos que -2 representa un pago anticipado y se conservaron sin modificar.
- **Desbalance de clases**: en el Problema B se identificó que solo el 22.1% de los clientes tiene default, lo que representa un reto para el modelado.

2. De realizar creación de variables, explica cuáles hiciste y por qué?

Se crearon variables en ambos problemas:

*Problema A (PAY_AMT4):*
- `BILL_AMT_PROM`: promedio de facturas de abril y mayo, para suavizar el comportamiento de un solo mes.
- `PAY_AMT_PROM`: promedio de pagos previos, captura el comportamiento histórico de pago.
- `RATIO_PAGO_MAY` y `RATIO_PAGO_ABR`: ratio pago/factura, captura qué tan responsable es el clente independientemente del monto absoluto.

*Problema B (default):*
- `MESES_RETRASO`: número de meses con retraso, resume el historial de incumplimientos.
- `MAX_RETRASO`: peor retraso registrado, captura el caso extremo del cliente.
- `BILL_PROM` y `PAY_PROM`: promedios de facturas y pagos para suavizar variaciones mensuales.
- `RATIO_PAGO_FACTURA`: tanto paga el cliente respecto a lo que debe.
- `USO_CREDITO`: qué tan cerca está del límite de crédito, señal de riesgo financiero.

3. De los modelos realizados, ¿cómo seleccionaste al mejor?

En ambos problemas se entrenó un modelo baseline (Regresión Lineal / Regresión Logística) y un modelo fuerte (XGBoost). La selección se basó en:
- **Problema A**: se priorizó R² y RMSE. XGBoost obtuvo R²=0.5979 vs 0.1544 del baseline, con un RMSE casi 30% menor. Además, el R² en cross-validation (0.6086) fue consistente con el resultado en test, descartando overfitting.
- **Problema B**: se priorizó ROC-AUC y GINI por el desbalance de clases. XGBoost obtuvo AUC=0.777 y GINI=0.555 vs 0.711 y 0.423 del baseline, siendo superior en todas las métricas.

En ambos casos XGBoost fue seleccionado gandor y mejor. por su capacidad de capturar relaciones no lineales, su robustez ante outliers y su mejor desempeño generalizado en validación cruzada.

4. ¿Qué desafíos encontraste y cómo los superaste?

- **Data leakage en PAY_AMT4**: el principal desafío fue identificar correctamente qué variables usar para predecir junio sin usar información de meses posteriores. Se resolvió mapeando explícitamente las fechas de cada variable y excluyendo todo lo posterior a junio.
- **Desbalance de clases en default**: el 77.9% de registros son no-default, lo que podría sesgar el modelo a predecir siempre 0. Se resolvió usando `scale_pos_weight` en XGBoost y `class_weight="balanced"` en Regresión Logística, y se priorizaron métricas como F1-Score y ROC-AUC sobre Accuracy.
- **Valores infinitos en feature engineering**: los ratios pago/factura generaron infinitos cuando el denominador era cero. Se resolvió sumando 1 al denominador y reemplazando infinitos con NaN antes del modelado.
- **Valores no documentados en el dataset**: los valores 0 en EDUCATION y MARRIAGE y el -2 en las columnas PAY requerían una decisión de negocio. Se resolvió marcando los ceros como NaN y conservando los -2 bajo el supuesto de pago anticipado.


### Limitaciones y Trabajo Futuro

- Los datos son de 2005, el comportamiento crediticio puede haber cambiado significativamente.
- No se exploró optimización de hiperparámetros con búsqueda exhaustiva (GridSearch/Optuna).
- Se podría explorar el uso de redes neuronales o LightGBM para comparación adicional.
- El umbral de clasificación podría optimizarse según el costo de negocio de cada tipo de error (falso positivo vs falso negativo).